# Multilingual Topic Modelling — Full Pipeline Demo

End-to-end walkthrough using the `multilingual_topic` library on synthetic data.

**Steps:**
1. Generate dummy multilingual corpus
2. Preprocess text
3. Machine translation (non-English → English)
4. Embed documents
5. Tune UMAP / HDBSCAN hyperparameters
6. Layer 1 — global topic model
7. Layer 2 — per-theme topic model + SetFit refinement
8. SetFit evaluation
9. Analysis and visualisation

In [ ]:
import pandas as pd
import numpy as np
from multilingual_topic import (
    TextPreprocessor,
    SentenceRepresentation,
    TopicModel,
    ParameterTuner,
    train_setfit,
    cross_validate_setfit,
    Evaluation,
    ClassifierValidationHelper,
    volume_over_time,
    Heatmap,
    top_n_distribution,
)
from datasets import Dataset

## Step 1 — Generate dummy multilingual corpus

In [ ]:
np.random.seed(42)
n = 500

templates = {
    'en': [
        'Sweden faces international criticism over religious freedoms',
        'Child welfare system under scrutiny in Muslim communities',
        'Diplomatic tensions rise following protest events in Stockholm',
        'Swedish government responds to disinformation campaign online',
        'International media coverage of Sweden grows significantly',
    ],
    'ar': [
        'السويد تواجه انتقادات دولية بشأن حرية الدين',
        'نظام رعاية الأطفال تحت المجهر في المجتمعات المسلمة',
        'توترات دبلوماسية تتصاعد في أعقاب احتجاجات ستوكهولم',
        'الحكومة السويدية ترد على حملة التضليل الإعلامي',
    ],
    'tr': [
        'İsveç dini özgürlükler konusunda uluslararası eleştiriyle karşılaşıyor',
        'Müslüman topluluklarda çocuk refahı sistemi inceleme altında',
        'Stockholm protestoları sonrası diplomatik gerilimler yükseliyor',
    ],
}

rows = []
for _ in range(n):
    lang = np.random.choice(['en', 'ar', 'tr'], p=[0.5, 0.35, 0.15])
    rows.append({
        'text':     np.random.choice(templates[lang]),
        'language': lang,
        'country':  np.random.choice(['Egypt','Turkey','Pakistan','Indonesia','Saudi Arabia']),
        'platform': np.random.choice(['Twitter','Facebook','Telegram']),
        'date':     pd.Timestamp('2022-01-01') + pd.Timedelta(days=int(np.random.randint(0, 600))),
    })

df = pd.DataFrame(rows)
print(f'{len(df)} posts | languages: {df.language.value_counts().to_dict()}')
df.head()

## Step 2 — Preprocessing

In [ ]:
pp = TextPreprocessor(
    apply_remove_links=True,
    apply_remove_mentions=True,
    apply_remove_hashtags=True,
    apply_remove_emojis=True,
)
df['text_clean'] = df['text'].apply(pp.preprocess)
df[['text', 'text_clean']].head(3)

## Step 3 — Machine translation

Non-English content is translated using mBART or Helsinki-NLP models.
Replace the simulation below with `translate_iterator` for real use.

In [ ]:
# Production usage:
# from multilingual_topic import ManyToManyTranslator, translate_iterator
# from multilingual_topic.translation import iter_df_as_dict
#
# translator = ManyToManyTranslator('mbart', model, tokenizer, max_length=512, use_gpu=True)
# translated = list(translate_iterator(
#     iter_df_as_dict(df[df.language != 'en']), batch_size=16, translator=translator,
#     text_col='text_clean', src_lang='ar_AR', tgt_lang='en_XX',
#     translated_col='text_en', translated_by_col='translator'
# ))

df['text_en'] = df.apply(
    lambda r: r['text_clean'] if r['language'] == 'en' else f'[translated] {r["text_clean"]}',
    axis=1
)
print(f'English: {(df.language=="en").sum()} | To translate: {(df.language!="en").sum()}')

## Step 4 — Embed documents

In [ ]:
# Use GeneralRepresentation for standard models, NomicRepresentation for nomic-embed-text
embedder = SentenceRepresentation(
    model_name='sentence-transformers/all-mpnet-base-v2',
    data=df,
    text_col='text_en',
    file_name='layer1_embeddings',
)
embedder.extract_embeddings(batch_size=32)
print(f'Embeddings: {embedder.embeddings.shape}')

## Step 5 — Tune UMAP / HDBSCAN hyperparameters

`ParameterTuner` lets you interactively tune UMAP and HDBSCAN settings,
visualise cluster structure, compute silhouette score, and save/load configurations.

In [ ]:
tuner = ParameterTuner(
    embeddings=embedder.embeddings,
    save_directory='./tuner_params',
)

# Tune UMAP
tuner.tune_umap(n_neighbors=15, n_components=5, min_dist=0.0)

# Tune HDBSCAN
tuner.tune_hdbscan(min_cluster_size=10, cluster_selection_epsilon=0.1)

# Apply and visualise — prints silhouette score and cluster scatter
tuner.apply_parameters()
tuner.visualize_parameters()

In [ ]:
# Save tuned parameters for reproducibility
tuner.save_parameters('layer1_params')

# Later: load them back
# tuner.load_parameters('layer1_params')

## Step 6 — Layer 1: Global topic model

In [ ]:
layer1 = TopicModel(data=df, text_col='text_en')
topics, probs = layer1.fit(embedder.embeddings)
df['topic_l1'] = topics
print('Layer 1 topic counts:')
print(df['topic_l1'].value_counts().head(10))

## Step 7 — Layer 2: Per-theme topic model + SetFit refinement

For each broad theme, embed the subset, tune parameters, and use SetFit
to steer the embedding space toward analytically useful cluster boundaries.

In [ ]:
# Select a theme to drill into
theme_df = df[df['topic_l1'] == 0].copy().reset_index(drop=True)
print(f'Theme 0: {len(theme_df)} documents')

# Simulate labels (in practice: annotate a sample with positive/negative examples)
theme_df['label'] = np.where(
    theme_df['text_en'].str.contains('diplomatic|criticism|government', case=False),
    'relevant', 'irrelevant'
)
print(theme_df['label'].value_counts().to_dict())

In [ ]:
# Build few-shot SetFit training dataset from cluster examples
helper = ClassifierValidationHelper(
    data=theme_df, theme_col='label', theme='relevant',
    src_text='text_en', text_col='text_en',
    label_col='label', prediction_col='pred', topic_col='topic_l1',
    sample_size=8,
)
cleaned = helper.prepare_data()
train_df, valid_df = helper.split_data(cleaned)
train_ds, valid_ds = helper.convert_split_to_dataset(train_df, valid_df, sample_size=8)

# Train SetFit — refines embedding space for next BERTopic layer
setfit_model = train_setfit(
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    model_name='sentence-transformers/all-mpnet-base-v2',
    num_epochs=1, batch_size=8,
)
print('SetFit model trained — use refined embeddings as input for Layer 2 BERTopic')

## Step 8 — SetFit evaluation

In [ ]:
y_pred = setfit_model(valid_ds['text'])
y_true = valid_ds['label']

ev = Evaluation(y_true, y_pred)
ev.report()
print('Metrics:', ev.metrics())
ev.confusion_matrix().show()

In [ ]:
# Cross-validation (uncomment with sufficient labelled data)
# results = cross_validate_setfit(
#     theme_df, text_col='text_en', label_col='label',
#     model_name='sentence-transformers/all-mpnet-base-v2',
#     n_folds=5, batch_size=8, num_epochs=1,
# )
#
# Real-world results (10-fold CV, nomic-embed-text-v1):
# Accuracy: mean=0.861  F1: mean=0.854  (min=0.818, max=0.926)

## Step 9 — Analysis and visualisation

In [ ]:
# Volume over time by country
fig = volume_over_time(df, date_col='date', group_col='country', top_n=5,
                       title='Post volume by country')
fig.show()

In [ ]:
# Country × topic heatmap
hm = Heatmap(df, row_col='country', col_col='topic_l1')
hm.plot(title='Country × Topic (Layer 1)').show()

In [ ]:
# Platform distribution
top_n_distribution(df, col='platform', title='Posts by platform').show()